In [1]:
import finnhub
import os
import time
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

FROM_DATE = "2023-01-01"
TO_DATE = "2025-12-31"

data = finnhub_client.stock_usa_spending(symbol="AAPL", _from=FROM_DATE, to=TO_DATE)

In [2]:
print("=== TOP LEVEL KEYS ===")
for key in data.keys():
    print(f"  {key}")

print(f"\nSpending activities returned: {len(data['data'])}")

=== TOP LEVEL KEYS ===
  data
  symbol

Spending activities returned: 7


In [3]:
print("=== FIRST ACTIVITY ===")
if data["data"]:
    for k, v in data["data"][0].items():
        print(f"  {k}: {v}")
else:
    print("  No spending activities found for AAPL in this date range")

=== FIRST ACTIVITY ===
  symbol: AAPL
  recipientName: APPLE INC
  recipientParentName: APPLE INC
  country: USA
  totalValue: 13585
  outlayedAmount: 0
  obligatedAmount: 13585
  potentialAmount: 13585
  actionDate: 2025-09-30
  performanceStartDate: 2025-09-30
  performanceEndDate: 2025-11-29
  awardingAgencyName: Department of State
  awardingSubAgencyName: Department of State
  awardingOfficeName: U.S. EMBASSY PARIS
  performanceCountry: FRA
  performanceCity: 
  performanceCounty: 
  performanceState: 
  performanceZipCode: 
  performanceCongressionalDistrict: CA-17
  awardDescription: TELECOM CONSULAR PROJECT NEW DESK PHONE PART
  naicsCode: 334210
  permalink: https://www.usaspending.gov/award/CONT_AWD_19FR6325K1263_1900_-NONE-_-NONE-/
  lastModifiedDate: 2025-10-09


In [4]:
print("=== SPENDING ACTIVITIES SUMMARY ===")
if data["data"]:
    for activity in data["data"]:
        print(f"  {activity['actionDate']} | {activity['awardingAgencyName']} | totalValue: {activity['totalValue']} | {activity['awardDescription'][:50] if activity['awardDescription'] else 'N/A'}")
else:
    print("  No activities to display")

=== SPENDING ACTIVITIES SUMMARY ===
  2025-09-30 | Department of State | totalValue: 13585 | TELECOM CONSULAR PROJECT NEW DESK PHONE PART
  2025-09-29 | Agency for International Development | totalValue: 0 | APPLE PRODUCTS AND SERVICES
  2025-09-22 | Department of State | totalValue: 13598.11 | TELEPHONE DEVICES
  2025-09-15 | Department of State | totalValue: 12283.64 | SMARTPHONE
  2025-06-11 | Department of State | totalValue: 63555.61 | ICASS/PROG: IPHONE 13 PRO/PRO MAX, UNLOCKED 128 GB
  2025-05-16 | Department of Justice | totalValue: 7470 | CREDIT CARD PURCHASE POP DATES: 09/29/2025 TO 09/2
  2025-01-01 | Department of Justice | totalValue: 299 | APPLE DEVELOPER ACCOUNT RENEWAL


### ─────────────────────────────────────────────
### SECTION 2 — PRODUCTION RUN (ALL 60 COMPANIES)
### ─────────────────────────────────────────────

In [5]:
import finnhub
import os
import time
from collections import defaultdict
from dotenv import load_dotenv
load_dotenv()

finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

FROM_DATE = "2023-01-01"
TO_DATE   = "2025-12-31"

tickers = [
    # DEFENSE - High Lobby
    "LMT", "RTX", "NOC", "GD", "BA", "LHX", "LDOS", "HII", "BAESY", "SAIC",
    # DEFENSE - Low Lobby
    "TXT", "TDG", "HEI", "DRS", "KTOS", "AVAV", "MRCY", "CW", "MOG.A", "DCO",
    # ENERGY - High Lobby
    "XOM", "CVX", "COP", "OXY", "BP", "NEE", "D", "DUK", "HAL", "BKR",
    # ENERGY - Low Lobby
    "SLB", "VLO", "PSX", "EOG", "FANG", "DVN", "CTRA", "AR", "CHRD", "MTDR",
    # TECH - High Lobby
    "MSFT", "AMZN", "GOOGL", "IBM", "ORCL", "PLTR", "BAH", "CACI", "PSN", "CRM",
    # TECH - Low Lobby
    "AAPL", "META", "NVDA", "CSCO", "PANW", "CRWD", "SNOW", "DDOG", "NET", "TWLO"
]

duplicates = [t for t in tickers if tickers.count(t) > 1]
assert not duplicates, f"Duplicate tickers found: {duplicates}"

results = {}
errors  = []

# ── field-level tracking ───────────────────────────────────────────────────
wrapper_present  = defaultdict(int)
wrapper_null     = defaultdict(int)
wrapper_types    = defaultdict(set)

activity_present = defaultdict(int)
activity_null    = defaultdict(int)
activity_types   = defaultdict(set)

# ── activity count tracking ────────────────────────────────────────────────
activity_counts     = []
no_activity_tickers = []
total_activities    = 0

total = len(tickers)

for i, ticker in enumerate(tickers):
    try:
        data = finnhub_client.stock_usa_spending(symbol=ticker, _from=FROM_DATE, to=TO_DATE)

        if not data:
            errors.append((ticker, "empty response — no wrapper returned"))
            results[ticker] = {}
        else:
            results[ticker] = data

            # ── wrapper-level fields ──────────────────────────────────────
            for field, value in data.items():
                if field == "data":
                    continue
                if value is None or value == "":
                    wrapper_null[field] += 1
                else:
                    wrapper_present[field] += 1
                    wrapper_types[field].add(type(value).__name__)

            # ── activity-level fields ─────────────────────────────────────
            activities = data.get("data", [])
            if not activities:
                no_activity_tickers.append(ticker)
            else:
                activity_counts.append(len(activities))
                total_activities += len(activities)

                for activity in activities:
                    for field, value in activity.items():
                        if value is None or value == "":
                            activity_null[field] += 1
                        else:
                            activity_present[field] += 1
                            activity_types[field].add(type(value).__name__)

    except Exception as e:
        errors.append((ticker, f"API error: {str(e)}"))
        results[ticker] = {}

    if i < len(tickers) - 1:
        time.sleep(2)

# ─────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────
successful    = len([r for r in results.values() if r])
empty_tickers = [t for t, r in results.items() if not r]
with_activity = len(activity_counts)

print(f"✅ Successfully pulled:              {successful} / {total}")
print(f"❌ Errors:                           {len(errors)}")
print(f"📭 Empty responses:                  {empty_tickers if empty_tickers else 'None'}")
print(f"📋 Tickers with spending activity:   {with_activity} / {total}")
print(f"⭕ Tickers with no activity:         {len(no_activity_tickers)} / {total}")
if no_activity_tickers:
    print(f"   {no_activity_tickers}")
print()

if errors:
    print("─── Errors ───")
    for ticker, msg in errors:
        print(f"   {ticker}: {msg}")
    print()

# ─────────────────────────────────────────
# SECTION 1 — ACTIVITY COUNT DISTRIBUTION
# ─────────────────────────────────────────
if activity_counts:
    print("─── Activity Count Distribution (tickers with spending only) ───")
    print(f"  Total activities across all tickers: {total_activities}")
    print(f"  Min activities per ticker:           {min(activity_counts)}")
    print(f"  Max activities per ticker:           {max(activity_counts)}")
    print(f"  Avg activities per ticker:           {sum(activity_counts) / len(activity_counts):.1f}")
    print()

# ─────────────────────────────────────────
# SECTION 2 — WRAPPER FIELD DECISION TABLE
# ─────────────────────────────────────────
print("─── Wrapper Field Decision Table ───")
print(f"{'Field':<20} {'Present':>12} {'Null':>8} {'Types':<20} Recommendation")
print("─" * 85)

all_wrapper_fields = set(wrapper_present.keys()) | set(wrapper_null.keys())
for field in sorted(all_wrapper_fields):
    present    = wrapper_present.get(field, 0)
    null_count = wrapper_null.get(field, 0)
    types      = ", ".join(wrapper_types.get(field, {"unknown"}))
    rec        = "Optional" if null_count > 0 else "Required"
    print(f"{field:<20} {present:>9} seen  {null_count:>5} null   {types:<20} {rec}")

print()

# ─────────────────────────────────────────
# SECTION 3 — ACTIVITY FIELD DECISION TABLE
# ─────────────────────────────────────────
print("─── Activity Field Decision Table (per spending record) ───")
print(f"{'Field':<40} {'Present':>12} {'Null':>8} {'Types':<20} Recommendation")
print("─" * 100)

all_activity_fields = set(activity_present.keys()) | set(activity_null.keys())
for field in sorted(all_activity_fields):
    present    = activity_present.get(field, 0)
    null_count = activity_null.get(field, 0)
    types      = ", ".join(activity_types.get(field, {"unknown"}))
    rec        = "Optional  ← null in some activities" if null_count > 0 else "Required  ← never null"
    print(f"{field:<40} {present:>9} seen  {null_count:>5} null   {types:<20} {rec}")

✅ Successfully pulled:              60 / 60
❌ Errors:                           0
📭 Empty responses:                  None
📋 Tickers with spending activity:   45 / 60
⭕ Tickers with no activity:         15 / 60
   ['DRS', 'COP', 'HAL', 'BKR', 'EOG', 'FANG', 'DVN', 'CTRA', 'AR', 'CHRD', 'MTDR', 'NVDA', 'PANW', 'CRWD', 'SNOW']

─── Activity Count Distribution (tickers with spending only) ───
  Total activities across all tickers: 40465
  Min activities per ticker:           2
  Max activities per ticker:           2000
  Avg activities per ticker:           899.2

─── Wrapper Field Decision Table ───
Field                     Present     Null Types                Recommendation
─────────────────────────────────────────────────────────────────────────────────────
symbol                      60 seen      0 null   str                  Required

─── Activity Field Decision Table (per spending record) ───
Field                                         Present     Null Types                Reco